# The Halting Diagonal — Results

**Run date:** 2026-07-29
**Engine:** `ValaQuenta/modules/turing_diagonal/maths.py`
**Data:** none external — all values computed from definitions.

---

## Prediction scoreboard

Each prediction from `01_predictions.ipynb` is re-declared here and executed.
Nothing is reported that was not run in this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.turing_diagonal import maths as td
from ValaQuenta import zero_lattice as zl

import math, itertools
from fractions import Fraction
import numpy as np

print('engine   : ValaQuenta.modules.turing_diagonal')
print('python   :', sys.version.split()[0])

In [ ]:
# Re-declare the registered predictions (01_predictions defines, 03 runs).

def P1_diagonal_escape(n_max=16, n_random=2000):
    for n in range(1, n_max + 1):
        for diag in itertools.product((0, 1), repeat=n):
            D = [1 - d for d in diag]
            if any(D[i] == diag[i] for i in range(n)):
                return False
    rng = np.random.default_rng(0)
    for _ in range(n_random):
        n = int(rng.integers(2, 12))
        T = rng.integers(0, 2, size=(n, n))
        D = 1 - np.diag(T)
        if any(np.array_equal(D, T[i]) for i in range(n)):
            return False
    return True

def _D_series(m):
    return int(sum(Fraction((-1)**k, math.factorial(k))
                   for k in range(m+1)) * math.factorial(m))

def _D_recur(m):
    a, b = 1, 0
    if m == 0: return a
    if m == 1: return b
    for k in range(2, m+1):
        a, b = b, (k-1)*(b+a)
    return b

def P2_derangement_exact(m_max=60):
    if any(_D_series(m) != _D_recur(m) for m in range(m_max+1)):
        return False
    return all(_D_recur(m) == round(math.factorial(m)/math.e)
               for m in range(1, 18))

def P3_involution():
    I2 = np.eye(2)
    i_m = np.array([[0.0, -1.0], [1.0, 0.0]])
    if not np.array_equal(np.linalg.matrix_power(i_m, 2), -I2): return False
    if not np.array_equal(np.linalg.matrix_power(i_m, 4),  I2): return False
    sq = [zl.multiply(zl.e_k(k), zl.e_k(k))[0] for k in range(16)]
    neg = [k for k in range(16) if abs(sq[k]+1) < 1e-12]
    pos = [k for k in range(16) if abs(sq[k]-1) < 1e-12]
    return neg == list(range(1, 16)) and pos == [0]

def P4_crib_pruning(A=26, N=200_000, tol=0.02):
    rng = np.random.default_rng(20260606)
    cipher = rng.integers(0, A, size=N)
    for L in (4, 8, 12, 16, 20, 25):
        crib = rng.integers(0, A, size=L)
        n_align = N - L + 1
        surv = sum(1 for s in range(n_align)
                   if not np.any(cipher[s:s+L] == crib))
        obs, pred = surv/n_align, (1 - 1/A)**L
        if abs(obs - pred)/pred > tol:
            return False
    return True

PREDICTIONS = [
    ('P1', 'Diagonal escapes every table; check is O(n) on diag only', P1_diagonal_escape),
    ('P2', 'D_n exact by three independent routes',                    P2_derangement_exact),
    ('P3', 'Involution order 4, no fixed point; e_0 unique at +1',     P3_involution),
    ('P4', 'Derangement prunes crib alignments to (1-1/A)^L',          P4_crib_pruning),
]

print('=' * 72)
print('PREDICTION SCOREBOARD — The Halting Diagonal')
print('=' * 72)
results = []
for tag, desc, fn in PREDICTIONS:
    try:
        ok = bool(fn())
        verdict = 'CONFIRMED' if ok else 'FAILED'
    except Exception as exc:
        ok, verdict = False, f'FAULT: {exc.__class__.__name__}'
    results.append((tag, desc, ok, verdict))
    print(f'  {tag} [{verdict:>9}] {desc}')
print('-' * 72)
n_ok = sum(1 for *_, ok, _ in results if ok)
print(f'  Overall: {n_ok}/{len(results)} confirmed')
print('=' * 72)

## The claim, restated against the numbers

**The escape is constructive and cheap.** Verified exhaustively for orders
1..16. Because the property depends only on `diag(T)`, that enumeration covers
every table of those orders — at `n=16` that is 65,536 diagonals standing in
for `2²⁵⁶` tables.

**The count is exact.** Three independent routes to `D_n` agree on the integers,
with no floating point in the first two.

In [ ]:
print(f'{"n":>3} {"D_n":>22} {"D_n/n!":>20}')
for m in (1, 2, 5, 10, 20, 26):
    Dm, fm = _D_recur(m), math.factorial(m)
    print(f'{m:>3} {Dm:>22} {Dm/fm:>20.16f}')
print(f'{"":>3} {"1/e":>22} {1/math.e:>20.16f}')
print()
D26 = _D_recur(26)
print(f'D_26/26! - 1/e = {D26/math.factorial(26) - 1/math.e:.3e}')
print('At n=26 the two are the same float64.')

**The involution is the same object at every level.** Order 4, determinant
`+1`, no fixed vector; and in the sedenion exactly 15 basis elements square to
`−1` with `e₀` the unique fixed point.

In [ ]:
sq = [zl.multiply(zl.e_k(k), zl.e_k(k))[0] for k in range(16)]
neg = [k for k in range(16) if abs(sq[k] + 1) < 1e-12]
pos = [k for k in range(16) if abs(sq[k] - 1) < 1e-12]
print(f'derangements (e_k^2 = -1) : k = {neg[0]}..{neg[-1]}, count {len(neg)}')
print(f'fixed point  (e_k^2 = +1) : k = {pos}')
print()
print('Historical chain, one operation each:')
for who, what in [
    ('Cantor 1891',  'd[n] != s_n[n]        flip the diagonal bit'),
    ('Godel 1931',   'G  != provable(G)     flip the provability'),
    ('Turing 1936',  'D(D) != HALT(D,D)     flip the halting status'),
    ('Enigma 1932',  'P(x) != x             flip the letter mapping'),
]:
    print(f'  {who:<14} {what}')

## What this paper does not show

Stated plainly so that no reader has to infer it:

- **HALT is still undecidable.** Nothing here decides it. The result is that the
  *construction* inside the proof is finite and checkable — a statement about
  the proof, not about computability.
- **P1 is verified, not proved, at finite order.** The argument that it holds for
  all `n` is the one-line construction in `00_holcus_vision`; the code confirms
  it to `n = 16` and does not extend further by itself.
- **`prediction_diagonal_test()` is excluded.** The engine ships it, and it is
  *not* used by any prediction above. It decides "self-reference" by testing
  whether an English string contains substrings such as `'this statement'`,
  `'not'` or `'if'`. That is a keyword classifier. It has no decision procedure
  behind it, it would report a different answer for a sentence translated into
  another language, and treating its output as a decidability verdict would be
  unsupported. Recorded here rather than quietly omitted.

In [ ]:
# Shown once, labelled for what it is -- not used in any claim above.
r = td.prediction_diagonal_test('this statement is false')
print('input   :', r['prediction'])
print('depth   :', r['diagonal_depth'])
print('state   :', r['i_power_analysis']['state'])
print('matched :', r['self_reference']['keywords_hit'][:6])
print()
r2 = td.prediction_diagonal_test('cette phrase est fausse')   # same sentence, French
print('input   :', r2['prediction'])
print('depth   :', r2['diagonal_depth'])
print('state   :', r2['i_power_analysis']['state'])
print('matched :', r2['self_reference']['keywords_hit'])
print()
print('Same proposition, different verdict. It is a substring test on English,')
print('which is why it supports no claim in this paper.')

## Status

| Prediction | Verdict |
|---|---|
| P1 — diagonal escapes; check is `O(n)` on the diagonal alone | see scoreboard |
| P2 — `D_n` exact by three independent routes | see scoreboard |
| P3 — involution order 4; `e₀` the unique fixed point | see scoreboard |
| P4 — crib alignments prune to `(1−1/A)^L` | see scoreboard |

**Wiki:** written last, per protocol. Not written yet.

**Engine:** `ValaQuenta/modules/turing_diagonal/` — 5/5 equations ESTABLISHED,
5/5 run clean.